# Cure Kinetics (26 points)

#### Authors (fill in your names and group number)


This notebook serves to deepen your knowledge on cure kinetics and how to interpret experimental data to get a cure kinetics model that can be used to design the cure process of a resin in composites manufacturing as well as which assumptions are made to do so. As explained during the lectures, part of research in composites manufacturing concerns  analysing curing processes of thermoset polymers. Thermoset resins, such as epoxy, undergo a curing process driven by chemical reactions. These reactions result in the creation of covalent bonds among monomers, ultimately forming polymer chains. This curing process is quantified by the parameter known as the 'degree of cure,' denoted as $\alpha$. Initially, in the absence of any covalent bonds, the material consists solely of individual monomers, resulting in $\alpha = 0$. However, upon the addition of a hardener, the monomers initiate the formation of polymer chains through the establishment of covalent bonds. When all the monomers have become part of polymer chains through these covalent bonds, it is deemed that the degree of cure has reached its maximum, $\alpha = 1$

Chemical reactions occur through contact of two active groups belonging to different molecules (for example of epoxy and of a hardener). Elevated cure temperatures are essential to trigger and maintain the chemical reactions responsible for converting the thermoset into a fully cured state. Whenever resin and hardener molecules (after mixing) are provided with more kinetic energy, they are more likely to shift and collide with neighbouring molecules, increasing the chances of curing reactions. Additionally, when more time is given to two reactants, the probability of a necessary collision for bonding increases. As a result, these phenomena are highly influenced by resin chemistry, hardener reactivity, cure temperature, and the presence of inhibitors or accelerators.

Therefore, we can say that the degree of cure $\alpha$ is a function of both temperature and time. $$\alpha = f(T,t)$$

Cure kinetics is concerned with the rates of the chemical reactions in a curing process. This is relevant, since the cure kinetics can help predict the thermoset cure and therefore also determines the manufacturing process and influences the final material properties. When cure kinetics are understood, they can be used to predict the degree of cure $\alpha$ of a certain process. To make this type of prediction, DSC measurement data can be used.

Before starting to work on the questions and to achieve a better understanding of the topic, it is highly recommended for you to read this paper: Kailong Jin, William H. Heath, John M. Torkelson, Kinetics of multifunctional thiol-epoxy click reactions studied by differential scanning calorimetry: Effects of catalysis and functionality. Polymer. 2015; 81: 70-78.

### How to use this notebook

Most code in this notebook is **pre-written and provided**. Your task is to:

1. **Run all cells in order** before answering any questions.
2. **Inspect the outputs carefully**: plots, printed values and printed parameters are the raw material for your answers.
3. **Answer the questions** in the dedicated answer cells.

One task requires you to **write your own code**: Question 5.1. It is marked with an empty code cell.

> **Evaluate-and-correct questions:** an AI-generated answer is provided. It may contain one or more errors. Identify every incorrect or misleading statement, explain *why* it is wrong, and provide the correct explanation.


## Setup — run this cell first

In [2]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter
from scipy.integrate import trapezoid
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# File names — adjust paths if needed
FILE_120 = "LME_0_13_120.xlsx"
FILE_150 = "LME_0_12b_150.xlsx"
FILE_180 = "LME_0_11_180.xlsx"


---
## Question 1 — DSC Data Cleaning and Visualisation (3 points)

Three isothermal DSC experiments were performed at 120 °C, 150 °C and 180 °C.
The raw data files are in `[mW]` and must be normalised to `[W/g]`.

The cell below loads the data, applies a cleaning pipeline, and plots the normalised heat flow $H\ [\text{W/g}]$ vs time $t\ [\text{min}]$ for all three temperatures.

**Read the code comments carefully** — they describe each step of the cleaning pipeline.
After running the cell, answer the questions below.


In [17]:
# ── Load raw data ──────────────────────────────────────────────────────────
# Each file has a multi-row header. The isothermal data starts where the
# sample temperature has reached the setpoint. We select rows by index,
# verified against the measured sample temperature (col index 5).
#
# Column layout in Sheet1:
#   col 1 → Time [min]
#   col 2 → Unsubtracted Heat Flow [mW]  (raw signal)
#   col 4 → Programmed (setpoint) temperature [°C]
#   col 5 → Measured sample temperature [°C]

excel_120 = pd.read_excel(FILE_120, sheet_name='Sheet1', header=None)
excel_150 = pd.read_excel(FILE_150, sheet_name='Sheet1', header=None)
excel_180 = pd.read_excel(FILE_180, sheet_name='Sheet1', header=None)

# Sample weights (mg) — read from file header
sampleweight120 = 13.7
sampleweight150 = 10.4
sampleweight180 = 14.4

# Select rows where sample is at the isothermal setpoint temperature
# (rows where col 5 is within ±3 °C of the target — machine equilibration excluded)
delta_H180 = excel_180.iloc[170:14525, 2].tolist()
delta_H150 = excel_150.iloc[66:28865,  2].tolist()
delta_H120 = excel_120.iloc[125:86489, 2].tolist()

timestep120 = np.diff(excel_120.iloc[125:86489, 1])
timestep150 = np.diff(excel_150.iloc[66:28865,  1])
timestep180 = np.diff(excel_180.iloc[170:14525, 1])

# Trim delta_H to match the length of timestep
delta_H120 = delta_H120[:-1]
delta_H150 = delta_H150[:-1]
delta_H180 = delta_H180[:-1]

# ── Step 1: Savitzky-Golay smoothing ───────────────────────────────────────
# Preserves peak shape while removing high-frequency noise.
delta_H120 = savgol_filter(delta_H120, window_length=11, polyorder=3)
delta_H150 = savgol_filter(delta_H150, window_length=11, polyorder=3)
delta_H180 = savgol_filter(delta_H180, window_length=611, polyorder=3)

# Convert to numpy arrays
delta_H120 = np.array(delta_H120)
delta_H150 = np.array(delta_H150)
delta_H180 = np.array(delta_H180)
timestep120 = np.array(timestep120)
timestep150 = np.array(timestep150)
timestep180 = np.array(timestep180)

# ── Step 2: Convert units [mW] → [W/g] ────────────────────────────────────
# Heat flow [mW] × time step [s] = energy per sample [mJ]; dividing by the
# sample weight [mg] gives J/g per sample. The data are sampled every 1 s, so
# this number is also the heat flow in W/g, the unit used on the y-axis from here on.
# Factor of 60: convert the time step from minutes to seconds.
delta_H_120n = (delta_H120 * timestep120 * 60) / sampleweight120
delta_H_150n = (delta_H150 * timestep150 * 60) / sampleweight150
delta_H_180n = (delta_H180 * timestep180 * 60) / sampleweight180

# ── Step 3: Baseline drift correction ─────────────────────────────────────
# The signal should asymptote to zero at long times (no more heat released).
# A negative asymptote arises from calibration drift. We shift each signal
# up by the mean of its last 50 points so the tail converges to zero.
delta_H_120n = [i - sum(delta_H_120n[-50:]) / len(delta_H_120n[-50:]) for i in delta_H_120n]
delta_H_150n = [i - sum(delta_H_150n[-50:]) / len(delta_H_150n[-50:]) for i in delta_H_150n]
delta_H_180n = [i - sum(delta_H_180n[-50:]) / len(delta_H_180n[-50:]) for i in delta_H_180n]

# ── Step 4: Second smoothing pass (Savitzky-Golay + Gaussian) ─────────────
delta_H_120n = savgol_filter(delta_H_120n, window_length=11, polyorder=3)
delta_H_150n = savgol_filter(delta_H_150n, window_length=11, polyorder=3)
delta_H_180n = savgol_filter(delta_H_180n, window_length=611, polyorder=3)
delta_H_120n = gaussian_filter(delta_H_120n, sigma=2)
delta_H_150n = gaussian_filter(delta_H_150n, sigma=2)
delta_H_180n = gaussian_filter(delta_H_180n, sigma=2)

# ── Step 5: Clip negative values to zero ──────────────────────────────────
# Negative heat flow after curing is physically impossible (would imply
# de-curing). Remaining negatives are artefacts of smoothing/calibration.
delta_H_120n = np.clip(delta_H_120n, 0, None)
delta_H_150n = np.clip(delta_H_150n, 0, None)
delta_H_180n = np.clip(delta_H_180n, 0, None)

# ── Time arrays ────────────────────────────────────────────────────────────
time120 = excel_120.iloc[125:86488, 1].to_numpy(dtype=float)
time150 = excel_150.iloc[66:28864,  1].to_numpy(dtype=float)
time180 = excel_180.iloc[170:14524, 1].to_numpy(dtype=float)

# ── Plot ───────────────────────────────────────────────────────────────────
#%matplotlib qt
plt.figure(figsize=(10, 5))
plt.plot(time120, delta_H_120n, label='120 °C', color='steelblue')
plt.plot(time150, delta_H_150n, label='150 °C', color='darkorange')
plt.plot(time180, delta_H_180n, label='180 °C', color='seagreen')
plt.xlabel('t [min]')
plt.ylabel('H [W/g]')
plt.title('Normalised heat flow vs time (isothermal DSC)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### Q1 — Interpretation (answer below)

1. In Step 5, negative values are clipped to zero. A classmate argues: *"Setting negatives to zero is physically correct — curing can only release heat, never absorb it."*
   Is this reasoning complete? What does clipping actually hide, and does it affect the downstream degree-of-cure calculation?

2. The row selection `iloc[170:14525]` for the 180 °C experiment marks the start of the isothermal phase.
   A student group used `iloc[127:14527]` for the 180 °C dataset instead.
   Inspecting the raw spreadsheet, at row 127 the **measured sample temperature** (col 5) reads approximately 138 °C — the sample is still ramping up and has not yet reached the 180 °C setpoint.
   What specific error does starting at row 127 (rather than row 170) introduce into the heat flow signal, and how does it affect the computed $\Delta H_{\max}$ and all downstream values of $\alpha$?


1.The reasoning is correct, curing can't absorb heat, but it is not complete. This is because negative values are mostly caused by measurement noise even after filtering. So over a long time on the tail end of the rate of heat flux the average of the noise should average out to zero. Hence also the degree of cure doesn't increase. However by clipping out the negative values, the average ovber the horizontal asymptote now has a positive bias, meaning it "adds cure" where the resin should have already reached its maximum degree of cure. This is more prevalent with the samples cured at 120degC and 150degC as their data has long assimptotes, over which this postive bias can accumulate as area under the ure curve.

However this can't be fully removed because in the case of the 120degC cure there is a large negative peak before at the start, which would lead to a negative degree of cure being added at the beginning which is phisically impossible. This is because the data starts to be taken before the sample has reached isotherm temperature, so there is a large negative heat flux as the sample is still taking energy to warm up to the program set temperature. So one would still need to clip this negative data at the start or just start at a later datapoint where the sample and program isotherm temperature have actually equalized. Of course some minor cure also occurs as the sample is heating up, which isn't accounted for in any case.

---
## Question 2 — DSC and Curing Rate (2 points)

> **Evaluate-and-correct.** Read the AI-generated answer below. Identify every incorrect or misleading statement, explain why it is wrong, and provide the correct explanation.

---

**AI answer:**

*"In a DSC experiment, the instrument measures the heat flow into or out of a sample. Since thermoset curing is an exothermic process, heat is released as covalent bonds form between monomers and the hardener. This heat flow (dH/dt) is directly proportional to the rate of the chemical reaction, and therefore to the curing rate dα/dt. Because of this proportionality, integrating the heat flow curve over time gives the total heat released, which can be normalised to obtain the degree of cure α.*

*The peak in the heat flow curve corresponds to the point at which the degree of cure α reaches its maximum value. Before the peak, the reaction accelerates because the rising temperature increases molecular mobility. After the peak, reactive groups are increasingly consumed, so the rate declines."*

---


**YOUR ANSWER HERE** *(double-click to edit)*

## Question 3 (4 points)

The isothermal DSC data can be used to estimate the degree of cure $\alpha$ as a function of time. The key assumption is that the total heat released during curing is proportional to the number of bonds formed:

$$\Delta H_{\mathrm{max}} \equiv \alpha = 1$$

where $\Delta H_{\mathrm{max}}$ is the **maximum total heat of reaction** observed across the three measurements:

$$\Delta H_{\mathrm{max}} = \max\!\left(\Delta H_{\mathrm{max},120}, \;\Delta H_{\mathrm{max},150}, \;\Delta H_{\mathrm{max},180}\right)$$

This assumption allows the heat flow to be normalised into a curing rate:

$$\frac{d\alpha}{dt} = \frac{1}{\Delta H_{\mathrm{max}}} \frac{dH(t)}{dt}$$

Integrating this over time gives $\alpha(t)$. The code below implements this procedure. Run it, inspect the resulting curves, and then work through the reflection questions that follow.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, trapezoid

# ── Use the cleaned heat-flow data from Question 1 ────────────────────────
# Time in minutes, heat flow in W/g.
t_120, H_120 = time120, np.asarray(delta_H_120n)
t_150, H_150 = time150, np.asarray(delta_H_150n)
t_180, H_180 = time180, np.asarray(delta_H_180n)

# ── Step 1: compute total heat released for each cure temperature ──────────
# Integrate each heat-flow curve using the trapezoidal rule.
# Result is in (W/g)·min, which is 1/60 of J/g. The constant factor cancels when
# we normalise by ΔH_max below, so α is unaffected.
dH_120 = trapezoid(H_120, t_120)
dH_150 = trapezoid(H_150, t_150)
dH_180 = trapezoid(H_180, t_180)

print(f'ΔH_max,120 = {dH_120:.3f}')
print(f'ΔH_max,150 = {dH_150:.3f}')
print(f'ΔH_max,180 = {dH_180:.3f}')

# ── Step 2: select ΔH_max from the three measurements ────────────────────
dH_max = max(dH_120, dH_150, dH_180)   # highest observed value
print(f'\nΔH_max used for normalisation = {dH_max:.3f}')

# ── Step 3: compute curing rate dα/dt and degree of cure α(t) ────────────
# Normalise heat flow → curing rate
dadt_120 = H_120 / dH_max
dadt_150 = H_150 / dH_max
dadt_180 = H_180 / dH_max

# Integrate curing rate → degree of cure  (initial condition α(t=0) = 0)
alpha_120 = cumulative_trapezoid(dadt_120, t_120, initial=0)
alpha_150 = cumulative_trapezoid(dadt_150, t_150, initial=0)
alpha_180 = cumulative_trapezoid(dadt_180, t_180, initial=0)

# ── Step 4: plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_120, alpha_120, label='120 °C')
ax.plot(t_150, alpha_150, label='150 °C')
ax.plot(t_180, alpha_180, label='180 °C')
ax.set_xlabel('t [min]')
ax.set_ylabel(r'Degree of cure $\alpha$ [-]')
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color='k', linestyle='--', linewidth=0.8, label=r'$\alpha = 1$ (full cure)')
ax.legend()
ax.set_title(r'Degree of cure $\alpha$ vs. time — standard $\Delta H_{\mathrm{max}}$')
plt.tight_layout()
plt.show()


#### Reflection on $\Delta H_{\mathrm{max}}$ and model validity

Before moving on, think carefully about the normalisation you just applied.

**3a.** The code above sets $\Delta H_{\mathrm{max}}$ to the largest integrated heat release observed across the three DSC measurements. Is this necessarily the true maximum heat of reaction for the resin, or is it the maximum *within our dataset*? What could make the true $\Delta H_{\mathrm{max}}$ of the resin differ from the largest value we measured here, and in which direction?


**YOUR ANSWER HERE** *(double-click to edit)*

**3b — Exploration.** To see how the choice of $\Delta H_{\mathrm{max}}$ affects your results, run the cell below. It repeats the calculation with a $\Delta H_{\mathrm{max}}$ that is **15 % higher** than the one you computed from the data and plots the new $\alpha(t)$ curves alongside the original ones. Then answer questions 3c–3e.


In [ ]:
# ── Exploration: effect of a higher ΔH_max ───────────────────────────────
# Suppose the true heat of reaction is 15 % higher than what we observed.
dH_max_higher = dH_max * 1.15

alpha_120_h = cumulative_trapezoid(H_120 / dH_max_higher, t_120, initial=0)
alpha_150_h = cumulative_trapezoid(H_150 / dH_max_higher, t_150, initial=0)
alpha_180_h = cumulative_trapezoid(H_180 / dH_max_higher, t_180, initial=0)

fig, ax = plt.subplots(figsize=(8, 5))
# Original
ax.plot(t_120, alpha_120, 'C0-',  label='120 °C — original ΔH_max')
ax.plot(t_150, alpha_150, 'C1-',  label='150 °C — original ΔH_max')
ax.plot(t_180, alpha_180, 'C2-',  label='180 °C — original ΔH_max')
# Higher ΔH_max
ax.plot(t_120, alpha_120_h, 'C0--', label='120 °C — ΔH_max × 1.15')
ax.plot(t_150, alpha_150_h, 'C1--', label='150 °C — ΔH_max × 1.15')
ax.plot(t_180, alpha_180_h, 'C2--', label='180 °C — ΔH_max × 1.15')
ax.set_xlabel('t [min]')
ax.set_ylabel(r'Degree of cure $\alpha$ [-]')
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color='k', linestyle=':', linewidth=0.8)
ax.legend(fontsize=8)
ax.set_title(r'Effect of $\Delta H_{\mathrm{max}}$ choice on $\alpha(t)$')
plt.tight_layout()
plt.show()


**3c.** Look at the two sets of curves. In which direction do all $\alpha$ values shift when a higher $\Delta H_{\mathrm{max}}$ is used, and why? Does the *shape* of the curves change, or only their magnitude?

**3d.** Look at where each curve ends. What does this imply about the **reported maximum degree of cure** at each temperature — is it a material property of the resin, or is it partly an artefact of the $\Delta H_{\mathrm{max}}$ choice? How does this uncertainty propagate when you use these curves to fit the Kamal-Sourour model in Question 5?

**3e.** Given this sensitivity, what would you recommend as best practice when reporting a degree-of-cure curve derived from isothermal DSC data?


**YOUR ANSWER HERE** *(double-click to edit)*

---
## Question 4 — Analysis of Degree of Cure (2 points)

> **Evaluate-and-correct.** Read the AI-generated answer below. Identify every incorrect or misleading statement, explain why it is wrong, and provide the correct explanation.

---

**AI answer:**

*"The graphs show that the maximum (plateau) degree of cure reached at the end of curing is essentially independent of cure temperature, provided that sufficient time is allowed. All three curves converge to the same final value, reflecting the fact that the total number of reactive functional groups in the resin system does not change with temperature. The differences in how quickly the plateau is reached are purely kinetic: higher temperatures simply accelerate the same reactions."*

---


**YOUR ANSWER HERE** *(double-click to edit)*

---
## Question 5 — Kamal-Sourour Cure Kinetics Model (7 points)

The Kamal-Sourour model predicts the curing rate as:

$$\frac{d\alpha}{dt} = (k_1 + k_2 \cdot \alpha^m)(1 - \alpha)^n$$

where $k_1$ and $k_2$ follow Arrhenius relationships:

$$k_i = A_i \cdot \exp\!\left(-\frac{E_i}{RT}\right), \quad i = 1, 2$$

### Question 5.1 — Estimating E₁ and A₁ (2 points)

Using the experimental isothermal DSC data from Question 1, estimate the initial activation energy $E_1$ and pre-exponential factor $A_1$ by plotting $\ln\!\left(\frac{d\alpha}{dt}\right)$ vs $\frac{1}{T}$ at low degree of cure ($\alpha \approx 0$) for the three temperatures. Use the data in the range $0.02 \le \alpha \le 0.12$ to determine $\frac{d\alpha}{dt}$ at each temperature. Calculate $E_1$ and $A_1$ from the slope and intercept.

Use the degree of cure $\alpha$ and time $t$ from Question 3 (the arrays `alpha_120`, `alpha_150`, `alpha_180` and `t_120`, `t_150`, `t_180`, with $t$ in minutes) and convert time to seconds so that the rates are in 1/s. Store your results in the variables `A1` (1/s) and `E1` (J/mol), because they are needed again in Question 5.2.


In [ ]:
# Question 5.1 — Arrhenius plot for k1


### Question 5.2 — Fitting A₂, E₂, m, n (3 points)

With E₁ and A₁ fixed from Question 5.1, the remaining four parameters (A₂, E₂, m, n) are optimised jointly against the 120 °C and 180 °C experimental data.


In [ ]:
# ── Prepare the data from Question 3 ──────────────────────────────────────
# Time is converted to seconds so that all rates are in 1/s (the same unit as in Question 8).
doc120, doc150, doc180 = alpha_120, alpha_150, alpha_180
time120b, time150b, time180b = t_120 * 60, t_150 * 60, t_180 * 60
R = 8.314  # J/(mol·K)

# ── Kamal-Sourour model ────────────────────────────────────────────────────
def kamal_sourour(alpha, T, A1, E1, A2, E2, m, n):
    k1 = A1 * np.exp(-E1 / (R * T))
    k2 = A2 * np.exp(-E2 / (R * T))
    return (k1 + k2 * np.abs(alpha)**m) * np.abs(1 - alpha)**n

# ── Experimental dα/dt from numerical gradient ─────────────────────────────
deriv_120 = np.gradient(doc120, time120b)
deriv_180 = np.gradient(doc180, time180b)

# Remove early transient artefacts (first points where α is very small and
# gradient is dominated by data-cleaning noise, not chemistry)
mask_120 = doc120 >= 0.012
mask_180 = doc180 >= 0.012

alpha_120_fit = doc120[mask_120]
alpha_180_fit = doc180[mask_180]
deriv_120_fit = deriv_120[mask_120][:-1]
deriv_180_fit = deriv_180[mask_180][:-1]

# Stack both datasets; pass temperature as a parallel input
T_120, T_180 = 120 + 273.15, 180 + 273.15
alphas_combined = np.concatenate([alpha_120_fit[:-1], alpha_180_fit[:-1]])
derivs_combined = np.concatenate([deriv_120_fit, deriv_180_fit])
temps_combined  = np.concatenate([T_120 * np.ones_like(alpha_120_fit[:-1]),
                                   T_180 * np.ones_like(alpha_180_fit[:-1])])

def combined_model(inputs, A2, E2, m, n):
    alpha, T = inputs
    return kamal_sourour(alpha, T, A1, E1, A2, E2, m, n)

initial_guess = [1e4, 60000, 0.6, 1.9]
bounds = ([0, 0, 0, 0], [1e6, 1e5, 3, 3])

popt, _ = curve_fit(
    combined_model,
    (alphas_combined, temps_combined),
    derivs_combined,
    p0=initial_guess,
    bounds=bounds,
    maxfev=10000
)
A2, E2, m_fit, n_fit = popt

# ── Check whether the optimiser pushed any parameter against its bound ────
names = ['A2', 'E2', 'm', 'n']
at_bound = [nm for nm, val, lo, hi in zip(names, popt, bounds[0], bounds[1])
            if abs(val - lo) < 1e-3 * (hi - lo) or abs(val - hi) < 1e-3 * (hi - lo)]
if at_bound:
    print("⚠ WARNING: the optimiser pushed " + ", ".join(at_bound) + " against the edge of its allowed range.")
    print("  The fit is then not reliable. Check your values of A1 and E1 from Question 5.1 (including their units)")
    print("  before interpreting the results below.\n")

print("Optimised Kamal-Sourour parameters:")
print(f"  A1 = {A1:.2e} 1/s   E1 = {E1:.1f} J/mol  (from Q5.1, fixed)")
print(f"  A2 = {A2:.2e} 1/s   E2 = {E2:.1f} J/mol")
print(f"  m  = {m_fit:.3f}   n  = {n_fit:.3f}")

# ── Plot: model vs experiment for 120 °C and 180 °C ──────────────────────
plt.figure(figsize=(10, 5))
for alpha_exp, deriv_exp, T_K, label, color in [
    (alpha_120_fit[:-1], deriv_120_fit, T_120, '120 °C', 'steelblue'),
    (alpha_180_fit[:-1], deriv_180_fit, T_180, '180 °C', 'darkorange'),
]:
    fitted = kamal_sourour(alpha_exp, T_K, A1, E1, A2, E2, m_fit, n_fit)
    plt.plot(alpha_exp, deriv_exp, linestyle='--', color=color, label=f'Experimental {label}')
    plt.plot(alpha_exp, fitted,    linestyle='-',  color=color, label=f'Model {label}')

plt.xlabel('Degree of cure α [-]')
plt.ylabel('dα/dt [1/s]')
plt.title('Kamal-Sourour model vs experiment (120 °C and 180 °C)')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### Q5.2 — Model critique (answer below)

1. The model was fitted to 120 °C and 180 °C data only. Looking at the plot, comment on how well the model fits each temperature and whether you notice any systematic discrepancy.

2. E₁ and A₁ were fixed from the Arrhenius estimate in Question 5.1 and were not re-optimised in the fit above. How could the uncertainty in E₁ and A₁ affect the fitted values of A₂, E₂, m and n? Would you expect the fit to change if E₁ and A₁ were also free parameters?


**YOUR ANSWER HERE** *(double-click to edit)*

### Question 5.3 — Model Validation at 150 °C (2 points)

A model fitted to two temperatures must generalise to unseen data. The 150 °C dataset was deliberately excluded from the fit. The cell below overlays the model prediction with the experimental 150 °C data.


In [ ]:
T_150 = 150 + 273.15

deriv_150 = np.gradient(doc150, time150b)
mask_150  = doc150 >= 0.03
alpha_150_val = doc150[mask_150][:-1]
deriv_150_val = deriv_150[mask_150][:-1]

fitted_150 = kamal_sourour(alpha_150_val, T_150, A1, E1, A2, E2, m_fit, n_fit)

plt.figure(figsize=(10, 5))
plt.plot(alpha_150_val, deriv_150_val, linestyle='--', color='purple', label='Experimental 150 °C')
plt.plot(alpha_150_val, fitted_150,    linestyle='-',  color='purple', label='Model 150 °C (validation)')
plt.xlabel('Degree of cure α [-]')
plt.ylabel('dα/dt [1/s]')
plt.title('Kamal-Sourour model — validation at 150 °C (not used in fit)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### Q5.3 — Comment (answer below)

1. Does the model accurately predict the 150 °C behaviour? Describe the agreement (or lack of it) quantitatively where possible (peak position, peak height, tail behaviour).

2. A student concludes: *"The model fits 150 °C well, which proves it is a correct physical description of the resin."* Critically evaluate this statement. What would you need to show before making such a claim?


**YOUR ANSWER HERE** *(double-click to edit)*

---
## Question 6 — Glass Transition Temperature (2 points)

> **Evaluate-and-correct.** Read the AI-generated answer below. Identify every incorrect or misleading statement, explain why it is wrong, and provide the correct explanation.

---

**AI answer:**

*"The glass transition temperature (Tg) is the temperature at which a polymer transitions from a hard, brittle glassy state to a soft, compliant state. Below Tg, polymer chain segments are effectively frozen in place and can only vibrate; they do not have enough thermal energy for large-scale cooperative motion. As temperature rises through Tg, chain segments gain sufficient energy to undergo cooperative segmental motion, leading to a dramatic change in mechanical properties such as stiffness and damping.*

*For thermoset polymers specifically, this transition results in the material entering a liquid-like state above Tg, where polymer chains can flow past one another. Because each additional crosslink formed during curing constrains the mobility of surrounding chain segments, an increased degree of cure leads to a lower Tg: more crosslinks mean more rigid segments that require less additional thermal energy to begin moving."*

---


**YOUR ANSWER HERE** *(double-click to edit)*

---
## Question 7 — Di Benedetto Equation (1 point)

The Di Benedetto equation is a mathematical empirical formula that relates the glass transition temperature ($T_g$) of a thermosetting polymer (such as an epoxy resin) to its degree of cure ($\alpha$):

$$\frac{T_g - T_{g0}}{T_{g\infty} - T_{g0}} = \frac{\lambda\,\alpha}{1-(1-\lambda)\,\alpha}$$

where $\lambda$ is a fitting parameter, $T_{g\infty}$ is the $T_g$ of the fully cured system, and $T_{g0}$ is the $T_g$ of the initial system.

**NOTE:** $T_{g0} = -54.577$ °C, $T_{g\infty} = 88.853$ °C.


In [ ]:
# ── Experimental Tg vs α data ─────────────────────────────────────────────
T_G0   = -54.577
T_G_inf =  88.853

alpha_Tg = np.array([0.049, 0.114, 0.242, 0.439, 0.645, 0.75, 0.9, 0.95, 0.98, 1.0])
T_G      = np.array([-48.633, -43.21, -30.083, -12.53, 14.733, 35.833, 57.5,
                      68.667, 80.333, 88.853])

# ── Fit λ ─────────────────────────────────────────────────────────────────
def dibenedetto(alpha, lam):
    return (T_G_inf - T_G0) * lam * alpha / (1 - (1 - lam) * alpha) + T_G0

popt_Tg, _ = curve_fit(dibenedetto, alpha_Tg, T_G, p0=[0.5])
lam_fit = popt_Tg[0]
print(f"Fitted λ = {lam_fit:.4f}")

# ── Plot ───────────────────────────────────────────────────────────────────
alpha_line = np.linspace(0, 1, 200)
Tg_model   = dibenedetto(alpha_line, lam_fit)

plt.figure(figsize=(7, 5))
plt.scatter(alpha_Tg, T_G, label='Experimental data', color='steelblue', zorder=5)
plt.plot(alpha_line, Tg_model, color='crimson',
         label=f'DiBenedetto fit (λ = {lam_fit:.4f})')
plt.xlabel('α [-]')
plt.ylabel('$T_g$ [°C]')
plt.title('Glass transition temperature vs degree of cure (Airstone 780E)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### Q7 — Report (answer below)

Report the fitted value of $\lambda$ and assess the quality of the fit visually.
Is the DiBenedetto equation a good description of this dataset across the full range of $\alpha$?
Are there regions where it fits better or worse?


**YOUR ANSWER HERE** *(double-click to edit)*

---
## Question 8 — Service Temperature and Process Design (5 points)

Two composite parts (A and B) made from Airstone 780E were cured isothermally at 80 °C:
- Part A: $\alpha = 0.90$
- Part B: $\alpha = 0.95$

### Question 8.1 — Service temperature of the two parts (1 point)


In [ ]:
def calc_Tg(alpha, lam=lam_fit):
    return (T_G_inf - T_G0) * lam * alpha / (1 - (1 - lam) * alpha) + T_G0

Tg_A = calc_Tg(0.90)
Tg_B = calc_Tg(0.95)

print(f"Part A (α = 0.90): Tg = {Tg_A:.1f} °C")
print(f"Part B (α = 0.95): Tg = {Tg_B:.1f} °C")
print(f"Difference:        ΔTg = {Tg_B - Tg_A:.1f} °C")


### Q8.1 — Comment (answer below)

The two parts differ by only 5 percentage points in degree of cure, yet their $T_g$ values differ by $\Delta T_g$ as printed above.

1. What does this difference mean in practical terms for the **service temperature** of each part?
2. Both parts were cured at 80 °C. Does the cure temperature of 80 °C impose any additional constraint on the maximum service temperature, regardless of $T_g$?
3. For an application requiring operation at 70 °C, which part would you recommend? Justify your choice.


**YOUR ANSWER HERE** *(double-click to edit)*

### Question 8.2 — Cure time at 80 °C (2 points)

The cure kinetics of Airstone 780E follows a modified Kamal model with an autocatalytic diffusion term:

$$\frac{d\alpha}{dt} = \frac{A \cdot \exp(-E/RT)}{1 + \exp\!\bigl(C(\alpha - \alpha_c - \alpha_T T)\bigr)}\,(1-\alpha)^n \cdot \alpha^m$$

with parameters:

| Parameter | Value |
|-----------|-------|
| A | 681085 s⁻¹ |
| E | 59291 J/mol |
| n | 1.67 |
| m | 0.12 |
| C | 47.7 |
| αc | 0.77 |
| αT | 0.0016 |



In [ ]:
# ── Model parameters ───────────────────────────────────────────────────────
A_mod  = 681085
E_mod  = 59291
n_mod  = 1.67
m_mod  = 0.12
C_mod  = 47.7
alpha_c = 0.77
alpha_T = 0.0016
R_mod   = 8.314

dt_ode = 0.01  # time step [s]

def dalpha(alpha, T_K):
    """Increment in α per time step dt_ode [s]."""
    rate = ((A_mod * np.exp(-E_mod / (R_mod * T_K))) /
            (1 + np.exp(C_mod * (alpha - alpha_c - alpha_T * T_K))))
    return rate * (1 - alpha)**n_mod * (alpha)**m_mod * dt_ode

T_80 = 80 + 273.15

# ── Cure to 90% (Part A) ───────────────────────────────────────────────────
alpha, t = 0.0001, 0
while alpha < 0.90:
    alpha += dalpha(alpha, T_80)
    t += dt_ode
tcA = t

# ── Cure to 95% (Part B) ───────────────────────────────────────────────────
alpha, t = 0.0001, 0
while alpha < 0.95:
    alpha += dalpha(alpha, T_80)
    t += dt_ode
tcB = t

print(f"Cure time to α = 0.90 (Part A): {tcA:.0f} s  ({tcA/60:.1f} min)")
print(f"Cure time to α = 0.95 (Part B): {tcB:.0f} s  ({tcB/60:.1f} min)")
print(f"Part B takes {tcB - tcA:.0f} s ({(tcB-tcA)/60:.0f} min) longer — "
      f"{(tcB/tcA - 1)*100:.0f}% more time for 5% more cure")


### Q8.2 — Reflection (answer below)

1. The cure time for Part B is disproportionately longer than for Part A, even though the target degree of cure increases by only 5 percentage points (0.90 → 0.95). Explain physically why the cure rate slows down so dramatically near full conversion in this model.

2. Considering the cure times and the $T_g$ values from Question 8.1, give a reasoned recommendation: should the production line target 90% or 95% cure at 80 °C? Include a discussion of the trade-off between production time and part performance.


**YOUR ANSWER HERE** *(double-click to edit)*

### Question 8.3 — Finding the Temperature for 60-minute Cure to 95% (2 points)

The production team wants to reach $\alpha = 0.95$ in exactly 60 minutes (3600 s).
The cell below searches for the required isothermal cure temperature with a **bisection search**: the cure time decreases as temperature increases, so the temperature interval between 80 °C and 200 °C is repeatedly halved until the temperature that gives a cure time of 3600 s is known to within 0.1 °C.


In [ ]:
# ── Bisection search for the temperature that achieves α = 0.95 in 3600 s ─
def cure_time(T_K, alpha_target=0.95):
    """Time [s] to reach alpha_target at constant temperature T_K [K]."""
    alpha, t = 0.0001, 0
    while alpha < alpha_target:
        alpha += dalpha(alpha, T_K)
        t += dt_ode
    return t

T_low, T_high = 80 + 273.15, 200 + 273.15   # [K]: too slow at T_low, fast enough at T_high
while T_high - T_low > 0.1:                  # stop when the interval is narrower than 0.1 °C
    T_mid = 0.5 * (T_low + T_high)
    if cure_time(T_mid) > 3600:
        T_low = T_mid                        # too slow: the answer lies at a higher temperature
    else:
        T_high = T_mid                       # fast enough: the answer lies at a lower temperature

T_search = T_high
t_cure   = cure_time(T_search)

T_result_C = T_search - 273.15
print(f"Temperature needed: {T_result_C:.1f} °C")
print(f"Cure time achieved: {t_cure:.0f} s  ({t_cure/60:.1f} min)")
print(f"Tg at α = 0.95:     {calc_Tg(0.95):.1f} °C")
print(f"Tg_inf (fully cured): {T_G_inf:.1f} °C")
print(f"Cure temperature vs Tg_inf: {T_result_C - T_G_inf:+.1f} °C")


### Q8.3 — Comment (answer below)

1. Report the cure temperature found and comment on how it compares to $T_{g\infty}$ of the fully cured resin. What does this proximity imply for the curing process?

2. The production team is excited — the model says the temperature works. Identify **at least two physical phenomena or practical concerns** that the model does not account for, and that you would flag before recommending implementation.


**YOUR ANSWER HERE** *(double-click to edit)*

---

## Group Reflection

Reflect as a group and discuss the following two questions. Write your answers below — a few sentences each is enough. There are no right or wrong answers here; what matters is that the response is genuinely yours.

**What was the most surprising result you encountered in this assignment?**

This could be something that contradicted your initial expectation, a number that seemed too large or too small, a curve that behaved differently from what the theory would suggest, or a point where the model broke down in a way you did not anticipate.

---
*Group answer:*



---

**If you could extend this analysis, what would you want to investigate further — and why?**

Think about limitations you noticed in the dataset, assumptions you were asked to accept without questioning, or a follow-up question that naturally arose while working through the problems.

---
*Group answer:*


---
## Declaration on the use of generative AI

Using generative AI tools to complete this assignment is **not allowed**, as stated in the course information. (The AI-generated answers provided in the evaluate-and-correct questions are part of the assignment and are not covered by this rule.)

**Each group member** confirms the declaration below by typing their name and replacing ☐ with ☒ (double-click this cell to edit it). Delete the lines you do not need.

☐ Name: ____________________ I declare that I did not use generative AI tools to complete this notebook, that I contributed to the answers submitted by my group, and that I can explain every answer if asked.

☐ Name: ____________________ I declare that I did not use generative AI tools to complete this notebook, that I contributed to the answers submitted by my group, and that I can explain every answer if asked.

☐ Name: ____________________ I declare that I did not use generative AI tools to complete this notebook, that I contributed to the answers submitted by my group, and that I can explain every answer if asked.

If you cannot truthfully confirm this declaration, do not submit the notebook before contacting the lecturer.
